# Kaggle 推理提交内核(自动生成,勿手改)
由 tools/push_inference.py 渲染生成。

In [ ]:
# ========== INFERENCE CONFIGURATION ==========
EXP_NAME = "__EXP_NAME__"
CHILD_EXP_NAME = "__CHILD_EXP_NAME__"
COMPETITION = "__COMPETITION__"
# 留空 = 默认读取 /kaggle/input/exp-results-{exp}-{child}
RESULTS_DATASET = ""
SUBMISSION_FILE = "/kaggle/working/submission.csv"
# ==============================================

print(f"Inferring {EXP_NAME}/{CHILD_EXP_NAME} on {COMPETITION}")

In [ ]:
%cd /kaggle/working
!pip install -q pyyaml pandas scikit-learn pillow pydicom

In [ ]:
# 数据准备:比赛未挂载时下载小文件 CSV 到 /kaggle/working/comp_data,并改写 config 的 data_dir
import pathlib, subprocess, yaml

comp_root = pathlib.Path(f"/kaggle/input/competitions/{COMPETITION}")
if (comp_root / "test.csv").exists():
    print("competition data mounted:", comp_root)
else:
    dst = pathlib.Path("/kaggle/working/comp_data")
    dst.mkdir(parents=True, exist_ok=True)
    for f in ["test.csv", "test_series.csv", "sample_submission.csv"]:
        r = subprocess.run(["kaggle", "competitions", "download", "-c", COMPETITION,
                           "-f", f], capture_output=True, text=True)
        src = pathlib.Path(f)
        if r.returncode == 0 and src.exists():
            src.rename(dst / f)
            print("downloaded", f)
        else:
            print("skip", f, r.stderr.strip()[:120])
    cfg = yaml.safe_load(pathlib.Path("config.yaml").read_text(encoding="utf-8"))
    cfg["data_dir"] = str(dst)
    pathlib.Path("config.yaml").write_text(yaml.safe_dump(cfg), encoding="utf-8")
    print("data_dir set to", dst)

In [ ]:
!python infer.py --config config.yaml --exp $EXP_NAME --child $CHILD_EXP_NAME --competition $COMPETITION --submission $SUBMISSION_FILE

In [ ]:
import pandas as pd
sub = pd.read_csv(SUBMISSION_FILE)
print(sub.shape)
print(sub.head())
print("READY_TO_SUBMIT")